# Movie Data Project 

In this project, I will extract data from ***TMDB***'s movies API, perform transformations, and conduct data analaysis to derive insights.

In [1]:
# Import packages all necessary packages
from dotenv import load_dotenv
import os
import sys
import requests
import pandas as pd
import numpy as np
import ast
import matplotlib.pyplot as plt
from datetime import datetime


# adding the path of the src folder to import cleaning functions script
sys.path.append(os.path.abspath(os.path.join('..', 'src')))
sys.path.append('src')

import cleaning_functions as main


<hr>

## STEP 1: API Data Extraction

In [ ]:
''' load .env file in this directory'''
load_dotenv()


'''get api key and save the base url'''
api_key = os.getenv('TMDB_API_KEY')

url = 'https://api.themoviedb.org/3/movie'

In [ ]:
'''list of movies to fetch from API'''

movie_ids = [0, 299534, 19995, 140607, 299536, 597, 135397, 
             420818, 24428, 168259, 99861, 284054, 12445, 181808, 
             330457, 351286, 109445, 321612, 260513]

In [ ]:
'''fetch and save movies to movies variable'''

movies = main.api_extract(url, api_key, movie_ids)

In [ ]:
'''confirming if all movies were fetched successfully'''

print(f'Movie IDs: {len(movie_ids)} || Exctracted Items: {len(movies)}')


**Movie id 0 was not found.** All movies were fetched successfully and saved to *movies* variable.

<hr>

### 1a Exploring the fetched data

API data structure is typically in JSON format. I am expecting same for these fetched data. 
- Lists nested in dictionaries
- Dictionaries nested in lists

I will explore briefly the structure of this data

In [ ]:
'''Check first item in the movies list'''

movies[0]

Fetched data indeed fits the assumptions. Lists nested in Dictionaries. This provides a chance to have a glance at the headers(***keys***) of the data to check if all columns were fetched. 

In [ ]:
'''Checking the keys of the dictionaries'''

print(movies[0].keys())
print(f'\nThere are {len(movies[0].keys())} column headers in the dataset.')

<hr>

### 1b. Covert List to DataFrame

I will convert the movies data into a pandas dataframe to be more structured

In [ ]:
'''Convert lists to DataFrame'''

df_movies = pd.DataFrame(movies)

### 1c. Explore the DataFrame

In [ ]:
'''Explore dataframe'''

# column names
df_movies.columns

In [ ]:
# display all columns columns to none
pd.set_option('display.max_columns', None)


# Set maximum column width to 500
pd.set_option('display.max_colwidth', 50)


In [ ]:
'''Explore first 2 records'''

df_movies.head(2)

**There are many columns that have nested lists and dictionaries.**

This is data is as dirty is it can be. 

### 1d. Export Data
Export or save this unprocessed ***(raw)*** data to the *data/* directory

In [ ]:
'''Export file as csv to /data/raw directory'''

today = datetime.today().strftime('%Y-%m-%d')  # add datetime function to indicate day the file was saved


df_movies.to_csv(f'../data/raw/raw_csv_data_{today}.csv', index=False)

## STEP 2: Data Cleaning and Preprocessing

In this phase, I do the *data cleaning and processing*. From exploring the dataset, dropping irrelevant columns, filtering and more.

* The data was fetched from an API, therefore it is consistent with JSON formats. Nested data structures.
* Normally, I'd create a copy of the dataset and work with the copy just in case something goes wrong but in this case, I will not be doing that.

### 2a. Data Preparation and Cleaning

In [2]:
'''read raw data'''

df_movies = pd.read_csv('../data/raw/raw_csv_data_2025-04-18.csv')

In [3]:
'''Check header of the dataframe'''

df_movies.head(0)

,adult,backdrop_path,belongs_to_collection,budget,genres,homepage,id,imdb_id,origin_country,original_language,...,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count,credits


In [4]:
'''Check the number of columns'''

print(f'There are {len(df_movies.columns)} columns in the dataset')

There are 27 columns in the dataset


In [5]:
'''View first two records'''
df_movies.head(2)

,adult,backdrop_path,belongs_to_collection,budget,genres,homepage,id,imdb_id,origin_country,original_language,...,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count,credits
0,False,/7RyHsO4yDXtBv1zUU3mTpHeQ0d5.jpg,"{'id': 86311, 'name': 'The Avengers Collection...",356000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 878, ...",https://www.marvel.com/movies/avengers-endgame,299534,tt4154796,['US'],en,...,2799439100,181,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Avenge the fallen.,Avengers: Endgame,False,8.237,26235,"{'cast': [{'adult': False, 'gender': 2, 'id': ..."
1,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,"{'id': 87096, 'name': 'Avatar Collection', 'po...",237000000,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",https://www.avatar.com/movies/avatar,19995,tt0499549,['US'],en,...,2923706026,162,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Enter the world of Pandora.,Avatar,False,7.588,32147,"{'cast': [{'adult': False, 'gender': 2, 'id': ..."


### 2b. Drop Unused Columns

In [6]:
'''drop irrelevant columns'''

df_movies = df_movies.drop(columns=['adult', 'imdb_id', 'original_title', 'video', 
                                    'homepage','backdrop_path', 'origin_country'])

### 2c. Evaluate JSON columns

In [7]:
'''check columns with unusual structure'''

df_movies[['belongs_to_collection', 'genres', 'production_countries',
           'production_companies', 'spoken_languages', 'credits']].head(2)

,belongs_to_collection,genres,production_countries,production_companies,spoken_languages,credits
0,"{'id': 86311, 'name': 'The Avengers Collection...","[{'id': 12, 'name': 'Adventure'}, {'id': 878, ...","[{'iso_3166_1': 'US', 'name': 'United States o...","[{'id': 420, 'logo_path': '/hUzeosd33nzE5MCNsZ...","[{'english_name': 'English', 'iso_639_1': 'en'...","{'cast': [{'adult': False, 'gender': 2, 'id': ..."
1,"{'id': 87096, 'name': 'Avatar Collection', 'po...","[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'iso_3166_1': 'US', 'name': 'United States o...","[{'id': 444, 'logo_path': None, 'name': 'Dune ...","[{'english_name': 'English', 'iso_639_1': 'en'...","{'cast': [{'adult': False, 'gender': 2, 'id': ..."


In [8]:
'''Explore the columns briefly'''

df_movies[['belongs_to_collection', 'genres', 'production_countries', 'production_companies', 
           'spoken_languages', 'credits']].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   belongs_to_collection  16 non-null     object
 1   genres                 18 non-null     object
 2   production_countries   18 non-null     object
 3   production_companies   18 non-null     object
 4   spoken_languages       18 non-null     object
 5   credits                18 non-null     object
dtypes: object(6)
memory usage: 996.0+ bytes


There are only 2 null values in the columns, **belongs_to_collection** in specific. The others do not have any non-null values.

In [9]:
'''Check the type of the first value in the column.'''

print(df_movies[['belongs_to_collection','genres','production_companies',
                 'production_countries','spoken_languages', 'credits' ]].iloc[0])


belongs_to_collection    {'id': 86311, 'name': 'The Avengers Collection...
genres                   [{'id': 12, 'name': 'Adventure'}, {'id': 878, ...
production_companies     [{'id': 420, 'logo_path': '/hUzeosd33nzE5MCNsZ...
production_countries     [{'iso_3166_1': 'US', 'name': 'United States o...
spoken_languages         [{'english_name': 'English', 'iso_639_1': 'en'...
credits                  {'cast': [{'adult': False, 'gender': 2, 'id': ...
Name: 0, dtype: object


These data are all of *string* datatype because they were converted into the dataframe. But are obviously **Dictionaries**, and **Lists**.

* ***belongs_to_collection*** and ***credits*** are dictionaries
* The remaining four are all lists with dictionaries nested.

### 2d. Extract Key Data Points

***Clean JSON like columns and seperate multiple values with |***

In [10]:
''' 01. Extract Collection Names from belongs_to_collection'''

df_movies['franchise'] = df_movies.apply(
    lambda row: main.extract_from_column(row, 'belongs_to_collection') or 'Standalone', axis=1
)


'''02.Extract genre names from genres seperated by |'''

df_movies['genres'] = df_movies.apply(lambda row: main.extract_from_column(row, 'genres', is_list=True), axis=1) 



'''03. Update Production companies'''

df_movies['production_companies'] = df_movies.apply(
    lambda row: main.extract_from_column(row, 'production_companies', is_list=True), axis=1)


'''04. Update Production Countries'''
df_movies['production_countries'] = df_movies.apply(
    lambda row: main.extract_from_column(row, 'production_countries', is_list=True), axis=1) 


'''Update Spoken Languages Column'''
df_movies['spoken_languages'] = df_movies.apply(
    lambda row: main.extract_from_column(row, 'spoken_languages', is_list=True), axis=1) 



In [11]:
print('\nUpdated columns')
df_movies[['title', 'franchise', 'genres', 'production_companies', 'production_countries', 'spoken_languages']].head()


Updated columns


,title,franchise,genres,production_companies,production_countries,spoken_languages
0,Avengers: Endgame,The Avengers Collection,Adventure|Science Fiction|Action,Marvel Studios,United States of America,English|日本語|
1,Avatar,Avatar Collection,Action|Adventure|Fantasy|Science Fiction,Dune Entertainment|Lightstorm Entertainment|20...,United States of America|United Kingdom,English|Español
2,Star Wars: The Force Awakens,Star Wars Collection,Adventure|Action|Science Fiction,Lucasfilm Ltd.|Bad Robot,United States of America,English
3,Avengers: Infinity War,The Avengers Collection,Adventure|Action|Science Fiction,Marvel Studios,United States of America,English|
4,Titanic,Standalone,Drama|Romance,Paramount Pictures|20th Century Fox|Lightstorm...,United States of America,English|Français|Deutsch|svenska|Italiano|Pусский


Ideally, I'd create a new column for the extracted data. But I made an exception today

In [12]:
'''
    Checking the columns to find how many columns I have still. 
    Before the cleaning process, the dataset had 27 columns. 
'''
print(f'There are {len(df_movies.columns)} columns in the dataset after first phase of cleaning')


There are 21 columns in the dataset after first phase of cleaning


 <hr>

***Extract New Columns From Credits***

In [13]:
'''Examine the credits column'''
df_movies['credits'].head(1)

0    {'cast': [{'adult': False, 'gender': 2, 'id': ...
Name: credits, dtype: object

In [14]:
'''Extract cast, cast_size, crew_size, directors from the credits'''

#extract engineered columns into credits_info
credits_info = df_movies.apply(main.extract_credits, axis=1)


# add engineered columns to existing df_movies
df_movies = pd.concat([df_movies, credits_info], axis=1)


'''check out new columns'''

print('\nNew columns created')
df_movies[['title', 'cast_size', 'crew_size', 'directors', 'cast']].head()


New columns created


,title,cast_size,crew_size,directors,cast
0,Avengers: Endgame,105,593,Joe Russo|Anthony Russo,Robert Downey Jr.|Chris Evans|Mark Ruffalo|Chr...
1,Avatar,65,986,James Cameron,Sam Worthington|Zoe Saldaña|Sigourney Weaver|S...
2,Star Wars: The Force Awakens,182,257,J.J. Abrams,Harrison Ford|Mark Hamill|Carrie Fisher|Adam D...
3,Avengers: Infinity War,69,724,Anthony Russo|Joe Russo,Robert Downey Jr.|Chris Evans|Chris Hemsworth|...
4,Titanic,116,258,James Cameron,Leonardo DiCaprio|Kate Winslet|Billy Zane|Kath...


In [ ]:
# Extract credits information
def extract_credits(row):
    try:
        if pd.notna(row['credits']):
            # Convert string to dictionary
            credits_dict = ast.literal_eval(row['credits'])
            
            # Get cast members (top 10 by order)
            cast = sorted(credits_dict.get('cast', []), key=lambda x: x.get('order', float('inf')))[:10]
            main_cast = "|".join([actor['name'] for actor in cast])
            
            # Get cast size
            cast_size = len(credits_dict.get('cast', []))
            
            # Get crew size
            crew_size = len(credits_dict.get('crew', []))
            
            # Get directors
            crew = credits_dict.get('crew', [])
            directors = "|".join([member['name'] for member in crew if member.get('job') == 'Director'])
            
            return pd.Series({
                'cast_size': cast_size,
                'crew_size': crew_size,
                'directors': directors,
                'cast': main_cast
            })
    except:
        pass
    return pd.Series({
        'cast_size': 0,
        'crew_size': 0,
        'directors': "",
        'cast': ""
    })

# Apply the function and create new columns
credits_info = df_movies.apply(extract_credits, axis=1)
df_movies = pd.concat([df_movies, credits_info], axis=1)

print('\nNew columns created')
df_movies[['title', 'cast_size', 'crew_size', 'directors', 'cast']].head()

In [ ]:
#drop credits
df_movies = df_movies.drop(columns='credits')

In [ ]:
# check total columns after extraction and creating new columns
col_names = list(df_movies.head(0))

print(f'\n Number of Columns {len(col_names)}')
print(col_names)

In [ ]:
df_movies.info()

### e. Convert of columns and datatypes

In [ ]:
# convert movies_df to timedate
df_movies['release_date'] = pd.to_datetime(df_movies['release_date'])

In [ ]:
# round up popularity and vote average figure to 2 decimal place 
df_movies[['popularity', 'vote_average']] = df_movies[['popularity', 'vote_average']].round(2)

In [ ]:
# convert revenue and budget to million_usd
df_movies['revenue_million_usd'] = (df_movies['revenue']/1e6).round(2)
df_movies['budget_million_usd'] = (df_movies['budget']/1e6).round(2)

In [ ]:
# drop columns
df_movies.drop_duplicates()

In [ ]:
# Count non-NaN values in each row
print('\nNumber of non Nan values in each row')
df_movies.notna().sum(axis=1)

In [ ]:
# filter dataframe by 'realeased' status and drop status

df_movies = df_movies[df_movies['status'] == 'Released']

In [ ]:
# evaluating columns again
df_movies.head(2)

In [ ]:
#rename collection_names to franchise
df_movies['franchise'] = df_movies['collection_name']
df_movies = df_movies.drop(columns='collection_name')

In [ ]:
col_names = list(df_movies.head(0))

print(f'\n Number of Columns {len(col_names)}')
print(col_names)

In [ ]:
df_movies = df_movies.drop(columns=['status', 'budget', 'revenue'])

In [ ]:
#reorder columns and reset index
df_movies = df_movies[['id', 'title', 'tagline', 'release_date', 'genres', 'franchise',
'original_language', 'budget_million_usd', 'revenue_million_usd', 'production_companies',
'production_countries', 'vote_count', 'vote_average', 'popularity', 'runtime',
'overview', 'spoken_languages', 'poster_path', 'cast', 'cast_size', 'directors', 'crew_size']]

In [ ]:
# reset the index
df_movies = df_movies.reset_index(drop=True)

In [ ]:
# save cleaned data to /data/processed directory
df_movies.to_csv('../data/processed/processed_csv_data.csv', index=False)

# STEP 3: KPI Performance and Analysis

In [ ]:
df_movies.head(2)

In [ ]:
# highest revenue
highest_revenue = df_movies.sort_values(by='revenue_million_usd', ascending=False).head(3)


print('\nTop 3 Highest Revenue Movies')
highest_revenue

**3 Movies with the highest revenue generated are:**
* Avatar: Enter the World of Pandora
* Avengers: Endgame
* Titanic

In [ ]:
# highest budget analysis
highest_budget = df_movies.sort_values(by='budget_million_usd', ascending=False).head(3)

print('Top 3 Movies with High Budgets')
highest_budget[['title', 'budget_million_usd']]

**Top 3 Movies with High budgets**
* Avengers: Age of Ultron with 365M at the top
* Avengers: Endgame with 356M at second
* Avengers: Infinity War 300M

These top 3 movies are all part of the Marvel Avengers franchise. Marvel allocates a lot of money for the Avengers collections of movies.

In [ ]:
#highest_profit
df_movies['profit'] = df_movies['revenue_million_usd'] - df_movies['budget_million_usd']
highest_profit = df_movies.sort_values(by='profit', ascending=False).head(3)
highest_profit[['title','tagline','profit']]

**The top 3 movies with the highest profit**
* Avatar: Enter the world of Pandora tops wiith 2.6B dollars
* Avengers: Endgame comes second with 2.4B
* Titanic is third with 2B

In [ ]:
# lowest profit
lowest_profit = df_movies.sort_values(by='profit', ascending=True).head(3)

print('Movies with the lowest profits')
lowest_profit[['title', 'profit']]

**Lowest performing movies with the lowest profit**
* Avengers: Age of Ultron with the highest budget made the less profit with 1B
* Incredibles 2 made 1B too with over 200k dollars over Avengers
* Beauty and the Beast made 1.1B

In [ ]:
# highest Return on Investment for budget >= 10M

df_roi = df_movies[df_movies['budget_million_usd'] >= 10] 
df_roi['roi'] = (df_roi['revenue_million_usd'] / df_roi['budget_million_usd']).round(2)
highest_roi = df_roi.sort_values(by='roi', ascending=False).head(3)

print('Top 3 Movies with high Return on Investment')
highest_roi[['title','roi']]

**Top 3 movies with the highest return on investments**
* Avatar is one of the best performing movies. It made 12M return on investments.
* Titanic is second with 11M
* Jurassic World is third 11M. Just a few dollars from Titanic

In [ ]:
#lowest ROI
lowest_roi = df_roi.sort_values(by='roi', ascending=True).head(3)

print('Lowest Return on Investment')
lowest_roi[['title','roi']]

**Top 3 Movies with low return on investments:**
* Avengers might be the worst performing movie by Marvel in this dataset. With the highest budget, it still had the worst ROI with 3M
* Incredibles 2  was second with 6M
* The Lion king is at 3rd with 6M

In [ ]:
# most voted movies
most_voted = df_movies.sort_values(by='vote_count', ascending=False).head(3)

print('Top 3 movies with most votes')
most_voted[['title','vote_count']]

**Top 3 movies with most votes:**
* Avatar comes in first as a fans favorite with 32119 votes
* The Avengers, the first movie of the avengers franchise comes in third with 31k votes
* Avengers: Infinity war is third with 30k votes

In [ ]:
# highest rated movies with at least 10 votes
df_rated = df_movies[df_movies['vote_count'] >= 10]
highest_rated = df_rated.sort_values(by='vote_average', ascending=False).head(3)

print('Top 3 highest rated movies')
highest_rated[['title','vote_average']]

**Top 3 highest rated movies:**
* Avengers Endgame is the highest rated movie at 8.24
* Avengers Infinity War with the same ratings
* Harry Potter with 8

In [ ]:
# lowest rated movies
lowest_rated = df_rated.sort_values(by='vote_average', ascending=True).head(3)

print('Lowest rated movies')
lowest_rated[['title','vote_average']]

**Top 3 Lowest Rated Movies:**
* Jurassic World: Fallen Kingdom perfomed with an average user rating of 6.54 which is poor in movie terms
* Jurassic World franchise again at second with 6.69 rating. People were too scared of the dinosaurs
* Star Wars: last Jedi with 6.78

In [ ]:
# most popular movies
most_popular = df_movies.sort_values(by='popularity', ascending=False).head(3)


print('Top 3 most popular movies')
most_popular[['title','tagline','popularity']]

**Top 3 Popular Movies:**
* Titanic - sits on top of the charts 
* Avengers come in at second
* Thanos defeating the Avengers didn't make them popular enough to top the charts. They sit at third

### b. Advanced Filtering

In [ ]:
# Filter for Sci-Fi Action movies with Bruce Willis
actor_filter = df_movies[
    df_movies['genres'].str.contains('Science Fiction', case=False) &
    df_movies['genres'].str.contains('Action', case=False) &
    df_movies['cast'].str.contains('Bruce Willis', case=False)
]

# Sort by Rating descending
filtered_sorted = actor_filter.sort_values(by='vote_average', ascending=False)


print('Movies with Bruce Willis in an Action Movie')
filtered_sorted[['title','tagline']]

No movies in this dataset has Bruce Willis casted in an Action movie

In [ ]:
# Filter for movies with Uma Thurman and directed by Tarantino
actor_filter = df_movies[
    df_movies['cast'].str.contains('Uma Thurman', case=False) &
    (df_movies['directors'].str.lower() == 'quentin tarantino')
]

# Sort by runtime ascending
filtered_sorted = actor_filter.sort_values(by='runtime', ascending=True)


print('Movies with Uma Thuram casted and directed by Quentin Tarantino')
filtered_sorted[['title','tagline']]

There are no Quentin Tarantino movies with Uma Thurman casted

In [ ]:
# print franchised and Standalone movies
print(df_movies['franchise'].value_counts())

* Marvel's Avengers has 4 movies in the Avengers collection
* There are two Standalone movies 
* ... more


### i. Mean Revenue for both Groups

In [ ]:
# Calculate mean revenue for both groups
franchise_revenue = df_movies[df_movies['franchise'] != 'Standalone']['revenue_million_usd'].mean()
standalone_revenue = df_movies[df_movies['franchise'] == 'Standalone']['revenue_million_usd'].mean()

print(f'franchise revenue: {franchise_revenue:.2f}')

print(f'Standalone revenue: {standalone_revenue:.2f}')

* Franchised Movies made a revenue of over 1.5B dollars
* Standalone movies made over 1.7B dollars

### ii. Median Return on Investments for both groups

In [ ]:
# Calculate ROI first
df_movies['roi'] = (df_movies['revenue_million_usd'] - df_movies['budget_million_usd']) / df_movies['budget_million_usd']

# Calculate median ROI for both groups
franchise_roi = df_movies[df_movies['franchise'] != 'Standalone']['roi'].median()
standalone_roi = df_movies[df_movies['franchise'] == 'Standalone']['roi'].median()


print(f"Franchise Movies ROI: {franchise_roi:.2f}")
print(f"Standalone Movies ROI: {standalone_roi:.2f}")
print(f"Difference in ROI: {(franchise_roi - standalone_roi):.2f}")
print(f"Percentage Difference: {((franchise_roi - standalone_roi) / standalone_roi * 100):.2f}%")

* Franchised Movies made 6.79M returns on their investments
* Stadalone movies made over 8M returns 
* The percentage difference between both groups is -21.25%
* And the difference in ROI is -1.83M dollars

### iii. Mean Budget for both groups


In [ ]:
# Calculate mean budget for both groups
franchise_budget = df_movies[df_movies['franchise'] != 'Standalone']['budget_million_usd'].mean()
standalone_budget = df_movies[df_movies['franchise'] == 'Standalone']['budget_million_usd'].mean()


print(f"Franchise Movies: ${franchise_budget:.2f}M")
print(f"Standalone Movies: ${standalone_budget:.2f}M")
print(f"Difference: ${(franchise_budget - standalone_budget):.2f}M")
print(f"Difference %: {((franchise_budget - standalone_budget) / standalone_budget * 100):.2f}%")

### iv. Mean Popularity index for both groups

In [ ]:
# Calculate mean popularity for both groups
franchise_popularity = df_movies[df_movies['franchise'] != 'Standalone']['popularity'].mean()
standalone_popularity = df_movies[df_movies['franchise'] == 'Standalone']['popularity'].mean()


print(f"Franchise Movies: {franchise_popularity:.2f}")
print(f"Standalone Movies: {standalone_popularity:.2f}")
print(f"Difference: {(franchise_popularity - standalone_popularity):.2f}")
print(f"Difference %: {((franchise_popularity - standalone_popularity) / standalone_popularity * 100):.2f}%")

### b. Franchises and Directors

In [ ]:
# Filter out standalone movies
franchises = df_movies[df_movies['franchise'] != 'Standalone']

# Count movies per franchise
movies_per_franchise = franchises['franchise'].value_counts()

movies_per_franchise.head()

In [ ]:
# Calculate total budget per franchise - group by franchise
total_budget = franchises.groupby('franchise')['budget_million_usd'].sum().sort_values(ascending=False)

print("\nTotal Budget per Franchise (M USD):")
print(total_budget)

print("\nTop 5 Franchises by Total Budget:")
total_budget.head()

In [ ]:
# Calculate mean budget per franchise
mean_budget = franchises.groupby('franchise')['budget_million_usd'].mean().sort_values(ascending=False)

print("\nMean Budget per Franchise (M USD):")
print(mean_budget)


print("\nTop 5 Franchises by Mean Budget:")
mean_budget.head()

In [ ]:
# Calculate total revenue per franchise
total_revenue = franchises.groupby('franchise')['revenue_million_usd'].sum().sort_values(ascending=False)

print("\nTotal Revenue per Franchise (M USD):")
print(total_revenue)


print("\nTop 5 Franchises by Total Revenue:")
print(total_revenue.head())

In [ ]:
# Calculate mean revenue per franchise
mean_revenue = franchises.groupby('franchise')['revenue_million_usd'].mean().sort_values(ascending=False)

print("\nMean Revenue per Franchise (M USD):")
print(mean_revenue)
print("\nTop 5 Franchises by Mean Revenue:")
print(mean_revenue.head())

In [ ]:
# Calculate mean rating per franchise
mean_rating = franchises.groupby('franchise')['vote_average'].mean().sort_values(ascending=False)

print("\nMean Rating per Franchise:")
print(mean_rating)


print("\nTop 5 Franchises by Mean Rating:")
print(mean_rating.head())

In [ ]:
# Count movies per director
movies_per_director = df_movies['directors'].value_counts()

print("Number of Movies per Director:")
print(movies_per_director)

print("\nTop 5 Directors by Number of Movies:")
print(movies_per_director.head())

In [ ]:
# Calculate total revenue per director
director_revenue = df_movies.groupby('directors')['revenue_million_usd'].sum().sort_values(ascending=False)

print("\nTotal Revenue per Director (M USD):")
print(director_revenue)
print("\nTop 5 Directors by Total Revenue:")
print(director_revenue.head())

In [ ]:
# Calculate mean rating per director
director_rating = df_movies.groupby('directors')['vote_average'].mean().sort_values(ascending=False)

print("\nMean Rating per Director:")
print(director_rating)


print("\nTop 5 Directors by Mean Rating:")
print(director_rating.head())

## STEP 4: Visualization

In [ ]:
df_movies = pd.read_csv('../data/processed/processed_csv_data.csv')
df_movies.head(1)

In [ ]:
# revenue vs budget

plt.figure(figsize=(9, 5))
# Plot franchise movies
franchise = df_movies[df_movies['franchise'] != 'Standalone']
plt.scatter(franchise['budget_million_usd'], franchise['revenue_million_usd'], 
            color='blue', alpha=0.6, label='Franchise')

# Plot standalone movies
standalone = df_movies[df_movies['franchise'] == 'Standalone']
plt.scatter(standalone['budget_million_usd'], standalone['revenue_million_usd'], 
            color='red', alpha=0.6, label='Standalone')

plt.title('Revenue vs. Budget Trends')
plt.xlabel('Budget (Million USD)')
plt.ylabel('Revenue (Million USD)')
plt.legend()
plt.grid(True)
plt.show()

* Most of the highest revenue points belong to franchise movies
* Standalone films tend to cluster in the lower revenue zone, with only one pushing above the 2B dollars mark
* Some franchise films with very high budgets (300M–350M) didn't perform as well as others with more modest budgets


In [ ]:
# scatter plot for popularity vs rating
plt.figure(figsize=(9, 5))
# Plot franchise movies
plt.scatter(franchise['popularity'], franchise['vote_average'], 
            color='blue', alpha=0.6, label='Franchise')

# Plot standalone movies
plt.scatter(standalone['popularity'], standalone['vote_average'], 
            color='red', alpha=0.6, label='Standalone')

plt.title('Popularity vs. Rating')
plt.xlabel('Popularity')
plt.ylabel('Rating')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# line graph of yearly trends

# Extract year from release_date
df_movies['year'] = pd.to_datetime(df_movies['release_date']).dt.year

# Calculate yearly metrics
yearly_metrics = df_movies.groupby('year').agg({
    'revenue_million_usd': 'mean',
    'budget_million_usd': 'mean'
}).reset_index()


In [ ]:

plt.figure(figsize=(9, 5))
plt.plot(yearly_metrics['year'], yearly_metrics['revenue_million_usd'], 
         'b-o', label='Mean Revenue')
plt.plot(yearly_metrics['year'], yearly_metrics['budget_million_usd'], 
         'r-o', label='Mean Budget')

plt.title('Yearly Trends in Box Office Performance')
plt.xlabel('Year')
plt.ylabel('Million USD')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# standalone vs franchise

# Calculate metrics
franchise_metrics = df_movies[df_movies['franchise'] != 'Standalone'].agg({
    'revenue_million_usd': 'mean',
    'budget_million_usd': 'mean'
})

standalone_metrics = df_movies[df_movies['franchise'] == 'Standalone'].agg({
    'revenue_million_usd': 'mean',
    'budget_million_usd': 'mean'
})


In [ ]:

plt.figure(figsize=(7, 4))
x = range(2)
width = 0.35

plt.bar([i - width/2 for i in x], [franchise_metrics['revenue_million_usd'], 
                                   franchise_metrics['budget_million_usd']], 
        width, label='Franchise')
plt.bar([i + width/2 for i in x], [standalone_metrics['revenue_million_usd'], 
                                   standalone_metrics['budget_million_usd']], 
        width, label='Standalone')

plt.title('Franchise vs. Standalone Success Comparison')
plt.xlabel('Metric')
plt.ylabel('Million USD')
plt.xticks(x, ['Revenue', 'Budget'])
plt.legend()
plt.grid(True)
plt.show()